In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import AdaBoostClassifier,GradientBoostingClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, LGBMRegressor

In [2]:
# XGBOOST,LIGHTRBGM, CATBOOST, 

In [3]:
df = pd.read_csv("Vehicle (1).csv")
df.head()

,Comp,Circ,D.Circ,Rad.Ra,Pr.Axis.Ra,Max.L.Ra,Scat.Ra,Elong,Pr.Axis.Rect,Max.L.Rect,Sc.Var.Maxis,Sc.Var.maxis,Ra.Gyr,Skew.Maxis,Skew.maxis,Kurt.maxis,Kurt.Maxis,Holl.Ra,Class
0,95,48,83,178,72,10,162,42,20,159,176,379,184,70,6,16,187,197,van
1,91,41,84,141,57,9,149,45,19,143,170,330,158,72,9,14,189,199,van
2,104,50,106,209,66,10,207,32,23,158,223,635,220,73,14,9,188,196,saab
3,93,41,82,159,63,9,144,46,19,143,160,309,127,63,6,10,199,207,van
4,85,44,70,205,103,52,149,45,19,144,241,325,188,127,9,11,180,183,bus


In [4]:
X,y = df.drop("Class",axis=1), df["Class"]
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=26,test_size=0.3,stratify=y)

In [5]:
ohc = OneHotEncoder(sparse_output=False, drop="first").set_output(transform="pandas")
ct = make_column_transformer((ohc,make_column_selector(dtype_include=object)),
                             ("passthrough", make_column_selector(dtype_include=['int64','float64'])),
                             verbose_feature_names_out=False)
ct = ct.set_output(transform='pandas')

## GradientBoostingClassifier

In [7]:
lr = np.linspace(0.01,0.9,10)
max_depth = [None, 2, 3, 4]
n_estimators = [25, 50, 100]
scores = []
for l in tqdm(lr):
    for m in max_depth:
        for n in n_estimators:
            gdc = GradientBoostingClassifier(random_state=26,learning_rate=l, n_estimators= n, max_depth= m)
            gdc.fit(X_train,y_train)
            y_pred = gdc.predict(X_test)
            scores.append([l,m,n,accuracy_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns=["Learning Rate","Max Depth", "N", "Score"])
df_scores.sort_values("Score",ascending=False)

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [01:43<00:00, 10.32s/it]


,Learning Rate,Max Depth,N,Score
100,0.801111,2.0,50,0.783465
101,0.801111,2.0,100,0.779528
106,0.801111,4.0,50,0.779528
103,0.801111,3.0,50,0.779528
58,0.405556,4.0,50,0.775591
...,...,...,...,...
98,0.801111,NaN,100,0.681102
96,0.801111,NaN,25,0.681102
85,0.702222,NaN,50,0.661417
84,0.702222,NaN,25,0.661417


## XGBoostClassifier

In [10]:
from sklearn.preprocessing import LabelEncoder

# Encode string targets into integers (0, 1, 2, 3...)
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

In [11]:
lr = np.linspace(0.01,0.9,10)
max_depth = [None, 2, 3, 4]
n_estimators = [25, 50, 100]
scores = []
for l in tqdm(lr):
    for m in max_depth:
        for n in n_estimators:
            gdm = XGBClassifier(random_state=26,learning_rate=l, n_estimators= n, max_depth= m)
            gdm.fit(X_train,y_train_encoded)
            y_pred = gdm.predict(X_test)
            scores.append([l,m,n,accuracy_score(y_test_encoded, y_pred)])
df_scores = pd.DataFrame(scores, columns=["Learning Rate","Max Depth", "N", "Score"])
df_scores.sort_values("Score",ascending=False)

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:23<00:00,  2.39s/it]


,Learning Rate,Max Depth,N,Score
95,0.702222,4.0,100,0.795276
58,0.405556,4.0,50,0.795276
77,0.603333,2.0,100,0.791339
65,0.504444,2.0,100,0.791339
43,0.306667,3.0,50,0.791339
...,...,...,...,...
6,0.010000,3.0,25,0.704724
8,0.010000,3.0,100,0.692913
5,0.010000,2.0,100,0.692913
4,0.010000,2.0,50,0.688976


### LightGBM [Light Gradient Boosting Machine]

In [16]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

In [17]:
lr = np.linspace(0.01,0.9,10)
max_depth = [None, 2, 3, 4]
n_estimators = [25, 50, 100]
scores = []
for l in tqdm(lr):
    for m in max_depth:
        for n in n_estimators:
            gdm = LGBMClassifier(random_state=26,verbose = -1,learning_rate=l, n_estimators= n, max_depth= m)
            gdm.fit(X_train,y_train)
            y_pred = gdm.predict(X_test)
            scores.append([l,m,n,accuracy_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns=["Learning Rate","Max Depth", "N", "Score"])
df_scores.sort_values("Score",ascending=False)

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:06<00:00,  1.48it/s]


,Learning Rate,Max Depth,N,Score
84,0.702222,NaN,25,0.803150
76,0.603333,2.0,50,0.799213
52,0.405556,2.0,50,0.799213
65,0.504444,2.0,100,0.799213
112,0.900000,2.0,50,0.795276
...,...,...,...,...
1,0.010000,NaN,50,0.700787
8,0.010000,3.0,100,0.696850
6,0.010000,3.0,25,0.692913
7,0.010000,3.0,50,0.692913


## Catboost

In [15]:
lr = np.linspace(0.01,0.9,10)
max_depth = [None, 2, 3, 4]
n_estimators = [25, 50, 100]
scores = []
for l in tqdm(lr):
    for m in max_depth:
        for n in n_estimators:
            gdm = CatBoostClassifier(random_state=26, verbose= False, learning_rate=l, n_estimators= n, max_depth= m)
            gdm.fit(X_train,y_train)
            y_pred = gdm.predict(X_test)
            scores.append([l,m,n,accuracy_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns=["Learning Rate","Max Depth", "N", "Score"])
df_scores.sort_values("Score",ascending=False)

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:30<00:00,  3.06s/it]


,Learning Rate,Max Depth,N,Score
94,0.702222,4.0,50,0.783465
101,0.801111,2.0,100,0.779528
95,0.702222,4.0,100,0.779528
89,0.702222,2.0,100,0.775591
71,0.504444,4.0,100,0.775591
...,...,...,...,...
15,0.108889,2.0,25,0.681102
16,0.108889,2.0,50,0.681102
5,0.010000,2.0,100,0.669291
3,0.010000,2.0,25,0.653543
